In [1]:
!pip install slider osrparse kaggle pandas numpy matplotlib seaborn scikit-learn gdown unzip

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 11.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 23.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 32.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 39.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 46.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 60.8 MB/s  0:00:00m0:00:0100:01
  Created wheel for unzip: filename=unzip-1.0.0-py3-none-any.whl size=1298 sha256=1741bb93dea19052fc964b101120f6da3289aea80a0a164eee11e1a53b9d1bf4
  Stored in directory: /root/.cache/pip/wheels/80/dc/7a/f8af45b

In [ ]:
!gdown https://drive.google.com/uc?id=1RDnDAFbhPxAVWznPqezwjyS6CRMjiVRQ

Downloading...
From (original): https://drive.google.com/uc?id=171phvzuPfFlXZaD79fXax3x8T2blU3dI
From (redirected): https://drive.google.com/uc?id=171phvzuPfFlXZaD79fXax3x8T2blU3dI&confirm=t&uuid=790722c4-8d58-4e93-b778-e079fbe5ed08
To: /root/data-v7-final3.zip
 71%|███████████████████████████           | 4.12G/5.80G [06:04<02:23, 11.7MB/s]

In [5]:
!pip install unzip

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unzip: filename=unzip-1.0.0-py3-none-any.whl size=1298 sha256=9f6758defca490230f8e37d472b93fbef3ca5e6b128d0072a6b5a1fd9a347bd9
  Stored in directory: /root/.cache/pip/wheels/82/8a/fb/5dbbccb3ff8c380e15eab2dd36c5fb256450a128fceab648f5
Successfully built unzip


In [2]:
!unzip /root/dataset.zip -d osu-net-dataset

Archive:  /root/dataset.zip
   creating: osu-net-dataset/content/data-v8/
  inflating: osu-net-dataset/content/data-v8/data_24458_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_17232_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_21883_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_22741_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_15765_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_19433_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_11141_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_5827_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_1592_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_10986_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_13658_part0.npz  
  inflating: osu-net-dataset/content/data-v8/data_24136_part2.npz  
  inflating: osu-net-dataset/content/data-v8/data_10318_part0.npz  
  inflating: osu-net-dataset/content/data-v8

In [1]:
import numpy as np
import osrparse
from slider import Beatmap
from slider.beatmap import Circle, Slider, Spinner
from tqdm import tqdm
import os
import torch.optim as optim
import torch
import torch.nn as nn
import glob
import random
import numpy as np

# Constants
PLAYFIELD_W = 512
PLAYFIELD_H = 384
SAMPLE_RATE_HZ = 60
dt = 1000 / SAMPLE_RATE_HZ


In [2]:
def get_perfect_click_label(current_time, hit_objects):
    """
    Returns 1.0 if the button should be held down at this millisecond, 0.0 otherwise.
    Based on 'Perfect' play rules (Auto).
    """
    # Helper to clean time
    def to_ms(t):
        if hasattr(t, 'total_seconds'): return t.total_seconds() * 1000
        return t

    # We only need to check objects close to current_time.
    # Since hit_objects is sorted, we could optimize, but a quick scan is fine for offline processing.
    # Optimization: In the main loop, we usually track 'ho_index'. We can search around there.

    # Hit Window (300 / 100 / 50). Let's be strict: +/- 25ms for the initial click.
    HIT_WINDOW = 25.0

    for obj in hit_objects:
        start = to_ms(obj.time)
        end = to_ms(obj.end_time) if hasattr(obj, 'end_time') else start

        # Optimization: If object is way in the past, skip
        if end < current_time - 50: continue
        # Optimization: If object is way in the future, stop
        if start > current_time + 50: break

        # LOGIC:

        # 1. SLIDER / SPINNER (Hold Duration)
        if hasattr(obj, 'repeat') or isinstance(obj, Spinner):
            # We hold from slightly before start to exactly the end
            if (start - HIT_WINDOW) <= current_time <= end:
                return 1.0

        # 2. CIRCLE (Tap)
        else:
            # We 'hold' for a short burst to ensure the click registers
            if abs(current_time - start) <= HIT_WINDOW:
                return 1.0

    return 0.0

def to_ms(t):
    """Robustly converts timedelta or objects to float milliseconds."""
    if hasattr(t, 'total_seconds'):
        return t.total_seconds() * 1000
    return float(t)

def get_slider_position(obj, current_time):
    """
    Calculates the EXACT position of the slider ball at current_time (ms).
    Handles Repeats (Ping-Pong) and Curves perfectly using O(1) math.
    """
    start_ms = to_ms(obj.time)
    end_ms = to_ms(obj.end_time)

    # 1. Calculate how far along the slider we are (in time)
    elapsed = current_time - start_ms
    total_duration = end_ms - start_ms

    # Safety checks
    if total_duration <= 0: return obj.position.x, obj.position.y
    if elapsed < 0: return obj.position.x, obj.position.y
    if elapsed > total_duration:
        # Return the tail position (calculated at progress 1 or 0 depending on repeats)
        if obj.repeat % 2 == 1:
             pos = obj.curve(1.0)
        else:
             pos = obj.curve(0.0)
        return pos.x, pos.y

    # 2. Calculate the specific span duration (one single slide)
    span_duration = total_duration / obj.repeat

    # 3. Determine which "Lap" we are on (0 = first forward, 1 = first backward...)
    current_lap = int(elapsed / span_duration)
    current_lap = min(current_lap, obj.repeat - 1)

    # 4. Calculate Progress within the current span (0.0 to 1.0)
    time_in_span = elapsed - (current_lap * span_duration)
    progress = time_in_span / span_duration

    # 5. Handle "Ping-Pong" (Reversing)
    if current_lap % 2 == 1:
        progress = 1.0 - progress

    # 6. Get Exact Position from the Curve
    # The slider library caches the curve calculation, so this is fast.
    try:
        pos = obj.curve(progress)
        return pos.x, pos.y
    except:
        # Fallback if curve fails
        return obj.position.x, obj.position.y

def get_target_context(current_time, hit_objects, search_start_index):
    """
    Determines the target coordinates based on the current time.
    Returns: (tx, ty, time_delta, obj_type, new_index)
    """
    idx = search_start_index

    # Fast-forward to the active object
    while idx < len(hit_objects):
        obj = hit_objects[idx]
        start_ms = to_ms(obj.time)
        end_ms = to_ms(obj.end_time) if hasattr(obj, 'end_time') else start_ms

        # If object ends in the future, it's relevant
        if end_ms > current_time:
            break
        idx += 1

    if idx >= len(hit_objects): return 0.5, 0.5, 0, 0, idx, 0.5, 0.5 # Default next

    obj = hit_objects[idx]
    start_ms = to_ms(obj.time)

    # --- DETERMINE TYPE & TARGET ---
    if isinstance(obj, Slider) or hasattr(obj, 'repeat'):
        # SLIDER: Track the Ball
        obj_type = 1.0

        if current_time < start_ms:
            # Slider hasn't started -> Aim at Head
            tx = obj.position.x / PLAYFIELD_W
            ty = obj.position.y / PLAYFIELD_H
            time_delta = (start_ms - current_time) / 1000.0
        else:
            # Slider is active -> Track the Curve
            raw_x, raw_y = get_slider_position(obj, current_time)
            tx = raw_x / PLAYFIELD_W
            ty = raw_y / PLAYFIELD_H
            time_delta = 0.0 # We are ON the object

    elif isinstance(obj, Spinner):
        # SPINNER: Aim Center
        obj_type = 2.0
        tx, ty = 0.5, 0.5
        time_delta = (start_ms - current_time) / 1000.0

    else:
        # CIRCLE: Aim Position
        obj_type = 0.0
        tx = obj.position.x / PLAYFIELD_W
        ty = obj.position.y / PLAYFIELD_H
        time_delta = (start_ms - current_time) / 1000.0

    next_idx = idx + 1

    if next_idx < len(hit_objects):
        next_obj = hit_objects[next_idx]

        # We generally just want the head of the next object
        # to know the general direction of flow
        ntx = next_obj.position.x / 512
        nty = next_obj.position.y / 384
    else:
        # End of map: Next target is same as current (stay still)
        ntx = tx
        nty = ty

    # Return 7 values now
    return tx, ty, time_delta, obj_type, idx, ntx, nty

def process_pair(replay_path, beatmap_path):
    try:
        # Parse files
        replay = osrparse.Replay.from_path(replay_path)
        beatmap = Beatmap.from_path(beatmap_path)

        r_data = replay.replay_data
        if not r_data: return None, None

        # Extract Replay Data
        times = np.array([d.time_delta for d in r_data])
        cumulative_times = np.cumsum(times)
        xs = np.array([d.x for d in r_data])
        ys = np.array([d.y for d in r_data])
        keys = np.array([int(d.keys) for d in r_data])

        start_time = cumulative_times[0]
        end_time = cumulative_times[-1]

        duration = end_time - start_time

        # SAFETY CHECK 1: Too short?
        if duration < 1000: return None, None

        # SAFETY CHECK 2: Too long? (Corrupt file or Marathon map)
        if duration > 1000000:
            print(f"Skipping {os.path.basename(replay_path)}: Duration too long ({duration/1000}s)")
            return None, None

        # SAFETY CHECK 3: Timestamps array size check
        # Calculate size before allocating
        num_frames = int(duration / dt)
        if num_frames > 50000: # Limit to ~50k frames per map
             return None, None

        if end_time - start_time < 1000: return None, None

        # Resample to 60Hz
        timestamps = np.arange(start_time, end_time, dt)
        interp_x = np.interp(timestamps, cumulative_times, xs)
        interp_y = np.interp(timestamps, cumulative_times, ys)
        interp_keys = np.interp(timestamps, cumulative_times, keys)
        interp_keys = (interp_keys > 0.1).astype(float)

        features = []
        labels = []
        ho_index = 0
        hit_objects = beatmap.hit_objects() # Get list once

        for i, t in enumerate(timestamps):
            # 1. Current State
            curr_x = interp_x[i] / PLAYFIELD_W
            curr_y = interp_y[i] / PLAYFIELD_H

            # 2. Target Context
            tx, ty, t_delta, obj_type, new_idx, ntx, nty = get_target_context(t, hit_objects, ho_index)
            ho_index = new_idx

            delta_x = tx - curr_x
            delta_y = ty - curr_y

            # Input Vector: [cx, cy, tx, ty, dx, dy, t, type]
            input_row = [curr_x, curr_y, tx, ty, delta_x, delta_y, t_delta, obj_type, ntx, nty]

            # Label Vector: [Next_X, Next_Y, Click]
            # We predict where the player actually moved in the NEXT frame
            if i < len(timestamps) - 1:
                next_x = interp_x[i+1] / PLAYFIELD_W
                next_y = interp_y[i+1] / PLAYFIELD_H
                click = interp_keys[i+1]

                label_row = [next_x, next_y, click]

                features.append(input_row)
                labels.append(label_row)

        return np.array(features, dtype=np.float32), np.array(labels, dtype=np.float32)

    except Exception as e:
        # print(f"Skipping {os.path.basename(replay_path)}: {e}")
        return None, None

In [2]:
class OsuAimModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=1024, num_layers=3):
        super(OsuAimModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_size),
            nn.ReLU()
        )

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        # --- FIX: Unconstrained Velocity ---
        # No Tanh. No Sigmoid. Just raw speed.
        self.aim_head = nn.Linear(hidden_size, 2)

        self.click_head = nn.Linear(hidden_size, 1)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        lstm_out, new_hidden = self.lstm(emb, hidden)

        # Predicts raw adjustment to current position
        vel = self.aim_head(lstm_out)

        click_logits = self.click_head(lstm_out)
        return vel, click_logits, new_hidden

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import glob
import os
from tqdm import tqdm

# --- CONFIG ---
DATA_PATH = "/root/osu-net-dataset/content/data-v9/"
SEQ_LEN = 300

def load_optimized_dataset(data_path, seq_len):
    print(f"Scanning {data_path}...")
    files = glob.glob(os.path.join(data_path, "*.npz"))

    # We will flatten everything into two giant arrays.
    # This is much friendlier to the CPU cache and Linux memory management than a list of 20k arrays.
    feature_list = []
    label_list = []

    # We also store valid start indices directly mapping to the giant arrays.
    valid_starts = []

    current_offset = 0

    print("Loading and Indexing data...")
    for f in tqdm(files):
        try:
            with np.load(f) as data:
                feat = data['features'].astype(np.float32)
                lab = data['labels'].astype(np.float32)

                length = len(feat)
                if length <= seq_len + 10: continue

                # --- VECTORIZED PRE-FILTERING ---
                # Instead of checking loop 10 times during training, we find valid spots now.

                # 1. Spinner Check (Col 7 > 1.5)
                # 2. Break Check (Col 6 > 5.0)
                # Create a boolean mask: 1 = Bad, 0 = Good
                is_bad = (feat[:, 7] > 1.5) | (feat[:, 6] > 5.0)

                # We need a window of SEQ_LEN to be fully clean (all 0s).
                # We can use a rolling window trick or simple loop with numba,
                # but standard numpy striding is fast enough for offline loading.

                # Create a sliding window view of the bad mask
                # shape: (num_windows, seq_len)
                shape = (length - seq_len + 1, seq_len)
                strides = (is_bad.strides[0], is_bad.strides[0])
                windows = np.lib.stride_tricks.as_strided(is_bad, shape=shape, strides=strides)

                # Check if ANY frame in the window is bad
                # any(axis=1) returns True if the window has a bad frame
                bad_windows = windows.any(axis=1)

                # Get valid start indices (relative to this file)
                # We perform a stride of 5 to reduce memory usage slightly and reduce correlation,
                # but you can use stride 1 for maximum data.
                local_valid = np.where(~bad_windows)[0][::5]

                if len(local_valid) == 0: continue

                # Append to lists
                feature_list.append(feat)
                label_list.append(lab)

                # Store Global Indices
                # These indices point to the start of a valid sequence in the concatenated array
                valid_starts.append(local_valid + current_offset)

                current_offset += length

        except Exception:
            continue

    print("Concatenating Arrays (This may take a moment)...")
    # Merge into one giant contiguous block of RAM
    BIG_FEATURES = np.concatenate(feature_list)
    BIG_LABELS = np.concatenate(label_list)
    VALID_INDICES = np.concatenate(valid_starts)

    print(f"Dataset Loaded!")
    print(f"Total Frames: {len(BIG_FEATURES):,}")
    print(f"Valid Sequences: {len(VALID_INDICES):,}")

    return BIG_FEATURES, BIG_LABELS, VALID_INDICES

# Execute Load
# Ensure you have enough RAM (64GB is plenty for this)
ram_feats, ram_labels, valid_indices = load_optimized_dataset(DATA_PATH, SEQ_LEN)

Scanning /root/osu-net-dataset/content/data-v8/...
Loading and Indexing data...


  1%|          | 346/29827 [00:00<00:42, 691.52it/s]

100%|██████████| 29827/29827 [00:43<00:00, 684.03it/s]


Concatenating Arrays (This may take a moment)...
Dataset Loaded!
Total Frames: 219,607,094
Valid Sequences: 39,166,941


In [4]:
def get_batch_from_ram(dataset, batch_size, seq_len, input_size=10, output_size=3):
    x_batch = np.zeros((batch_size, seq_len, input_size), dtype=np.float32)
    y_batch = np.zeros((batch_size, seq_len, output_size), dtype=np.float32)

    dataset_size = len(dataset)

    i = 0
    while i < batch_size:
        # Pick a random replay from RAM (Instant)
        rand_idx = np.random.randint(0, dataset_size)
        features, labels = dataset[rand_idx]

        total_frames = len(features)

        # We already filtered short files during loading, so we can skip that check

        for attempt in range(10):
            start_idx = np.random.randint(0, total_frames - seq_len)

            # Get raw slice using slicing (Fast)
            # .copy() IS MANDATORY HERE to avoid modifying the master dataset
            x_slice = features[start_idx : start_idx + seq_len].copy()

            # --- SPINNER FILTER ---
            # If sequence contains a Spinner (Type 2.0), skip it
            # Assuming Object Type is Column 7
            if np.any(x_slice[:, 7] > 1.5): continue

            # --- BREAK FILTER ---
            # If Time Delta (Col 6) > 5s, skip
            if np.max(x_slice[:, 6]) > 5.0: continue

            time_noise = np.random.normal(0, 0.005, (seq_len, 1))
            x_slice[:, 6:7] += time_noise

            # Normalize Time (Col 6)
            x_slice[:, 6] = np.clip(x_slice[:, 6], 0.0, 1.0)

            # --- NOISE INJECTION ---
            # Add noise to Current Pos (Col 0, 1)
            noise = np.random.normal(0, 0.02, (seq_len, 2))
            x_slice[:, 0:2] += noise
            x_slice[:, 0:2] = np.clip(x_slice[:, 0:2], -0.5, 1.5)

            # Recalculate Deltas (Col 4, 5)
            # Delta = Target - Current
            x_slice[:, 4] = x_slice[:, 2] - x_slice[:, 0]
            x_slice[:, 5] = x_slice[:, 3] - x_slice[:, 1]

            # Get Labels
            next_pos_slice = features[start_idx+1 : start_idx + seq_len + 1, 0:2]
            next_pos_slice = np.clip(next_pos_slice, 0.0, 1.0)

            click_slice = labels[start_idx : start_idx + seq_len, 2:3]

            # Stack labels
            y_slice = np.concatenate([next_pos_slice, click_slice], axis=1)

            x_batch[i] = x_slice
            y_batch[i] = y_slice
            i += 1
            break

    return torch.from_numpy(x_batch), torch.from_numpy(y_batch)

In [5]:
class OsuIndexedDataset(Dataset):
    def __init__(self, features, labels, indices, seq_len):
        self.features = features
        self.labels = labels
        self.indices = indices
        self.seq_len = seq_len

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Direct lookup - O(1)
        start = self.indices[idx]
        end = start + self.seq_len

        # Slicing a numpy array is very fast (returns a view)
        # We perform .copy() here to ensure the resulting Tensor owns its memory
        # which is better for pinned_memory transfer.
        x = self.features[start:end].copy()

        # Get labels
        y_raw = self.labels[start:end]
        next_pos = y_raw[:, 0:2].copy()
        clicks = y_raw[:, 2:3].copy()

        return x, next_pos, clicks

In [ ]:
class ExponentialAsymmetricClickLoss(nn.Module):
    def __init__(self, pos_weight=15.0, base_penalty=2.0, expo_factor=5.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none', pos_weight=torch.tensor([pos_weight]))
        self.base_penalty = base_penalty
        self.expo_factor = expo_factor

    def forward(self, pred, target):
        # 1. Base BCE Loss
        loss = self.bce(pred, target)

        # 2. Identify "Late" Errors (Misses)
        # Target is 1 (Click), but Prediction is Low (< 0.5)
        # prob_error = 1.0 - sigmoid(pred)  (High when we miss a click)
        probs = torch.sigmoid(pred)
        prob_error = (target * (1 - probs))

        # 3. Exponential Penalty
        # If error is small (0.1), penalty is small.
        # If error is large (0.8), penalty explodes.
        # This focuses the model specifically on fixing the "Gross Misses" (Late Goods)
        # while being gentle on the "Near Misses" (Late Perfects).

        # dynamic_penalty = base * (1 + error^factor)
        dynamic_penalty = self.base_penalty * (1 + (prob_error ** self.expo_factor))

        # Apply
        weighted_loss = loss + (prob_error * dynamic_penalty)

        return weighted_loss.mean()

In [6]:
class AsymmetricClickLoss(nn.Module):
    def __init__(self, pos_weight=15.0, penalty_for_late=2.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none', pos_weight=torch.tensor([pos_weight]))
        self.penalty = penalty_for_late

    def forward(self, pred, target):
        # Calculate standard BCE
        loss = self.bce(pred, target)

        # --- ASYMMETRIC PENALTY ---
        # If we are LATE (Predicted < Target when Target is High), punish harder.
        # This forces the AI to bias towards "Early/On-Time".

        # Note: This is a simplification. A true temporal check is complex.
        # A simpler heuristic: Penalize "False Negatives" (Misses) harder than False Positives.

        # If Target is 1.0 and Pred is 0.0 (Miss/Late), multiply loss
        miss_penalty = (target * (1 - torch.sigmoid(pred))) * self.penalty

        return (loss + miss_penalty).mean()

In [5]:
def get_batch(file_list, batch_size, seq_len, input_size=10, output_size=3):
    x_batch = np.zeros((batch_size, seq_len, input_size), dtype=np.float32)
    y_batch = np.zeros((batch_size, seq_len, output_size), dtype=np.float32)

    i = 0
    while i < batch_size:
        idx = np.random.choice(len(file_list))
        try:
            data = np.load(file_list[idx])
            features = data['features']
            labels = data['labels']

            if len(features) <= seq_len + 10: continue

            for attempt in range(10):
                start_idx = np.random.randint(0, len(features) - seq_len)

                # Get raw slice
                x_slice = features[start_idx : start_idx + seq_len].copy()

                # --- IMPROVED FILTER: Check Object Type (Col 7) ---
                # 0=Circle, 1=Slider, 2=Spinner.
                # If the sequence contains a Spinner (2.0), skip it.
                if np.any(x_slice[:, 7] > 1.5): continue

                # --- FIX 1: BREAK FILTER & TIME NORM ---
                # Check Time Delta (Col 6). If > 5s, it's a break.
                if np.max(x_slice[:, 6]) > 5.0: continue
                # Normalize Time to max 1.0 (LSTM loves 0-1 range)
                x_slice[:, 6] = np.clip(x_slice[:, 6], 0.0, 1.0)

                # --- FIX 2: NOISE INJECTION (The Stability Fix) ---
                # Add small random noise to Current Position (Col 0, 1)
                # This simulates the AI being slightly off-target (~5-10 pixels)
                noise = np.random.normal(0, 0.02, (seq_len, 2))
                x_slice[:, 0:2] += noise

                # Sanitize Inputs (Keep on screen)
                x_slice[:, 0:2] = np.clip(x_slice[:, 0:2], -0.5, 1.5)

                # IMPORTANT: Recalculate Deltas (Col 4, 5)
                # Because we changed Current Position, the distance to Target changed!
                # Delta = Target - Current
                x_slice[:, 4] = x_slice[:, 2] - x_slice[:, 0]
                x_slice[:, 5] = x_slice[:, 3] - x_slice[:, 1]

                # Get Labels
                next_pos_slice = features[start_idx+1 : start_idx + seq_len + 1, 0:2]

                # Clamp Labels (Matches screen bounds)
                next_pos_slice = np.clip(next_pos_slice, 0.0, 1.0)

                click_slice = labels[start_idx : start_idx + seq_len, 2:3]
                y_slice = np.concatenate([next_pos_slice, click_slice], axis=1)

                x_batch[i] = x_slice
                y_batch[i] = y_slice
                i += 1
                break
        except: continue

    return torch.from_numpy(x_batch), torch.from_numpy(y_batch)

In [11]:
# --- CONFIG ---
BATCH_SIZE = 2048      # Increased from 64 for stable gradients
SEQ_LEN = 300     # 1 second of context is enough for physics
NUM_BATCHES = 100000   # INCREASED: Train for much longer
INITIAL_LR = 1e-3     # Start higher

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
# Re-init model
model = OsuAimModel(hidden_size=512).to(device)
optimizer = optim.Adam(model.parameters(), lr=INITIAL_LR)

# Check files
all_files = glob.glob(os.path.join("/root/osu-net-dataset/content/data-v6/", "*.npz"))
if len(all_files) == 0:
    raise ValueError(f"No files found. Please run the process_pair_v2 block!")
print(f"Found {len(all_files)} files in V2 folder.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


# scheduler reduces LR when loss stops dropping
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2000
)

pos_weight = torch.tensor([5.0]).to(device) # Emphasize clicking

# Keep L1 Loss for sharpness
aim_criterion = nn.L1Loss()
click_criterion = AsymmetricClickLoss(pos_weight=5.0, penalty_for_late=5.0).to(device)

print("Starting Long-Run Training (Slider Aware)...")

loss_history = []
pbar = tqdm(range(NUM_BATCHES), desc="Training Final Model", ncols=100)

for step in pbar:
    # Get Batch (New data v2)
    x_np, y_np = get_batch_from_ram(RAM_DATASET, BATCH_SIZE, SEQ_LEN)
    x = x_np.to(device)
    y = y_np.to(device)

    optimizer.zero_grad()

    # 1. Predict Velocity
    pred_vel, pred_click, _ = model(x)

    # 2. Physics Step (Residual)
    curr_pos = x[:, :, 0:2]
    pred_pos = curr_pos + pred_vel

    # 3. Calculate Loss
    true_pos = y[:, :, 0:2]
    loss_aim = aim_criterion(pred_pos, true_pos)
    loss_click = click_criterion(pred_click, y[:, :, 2:3])

    total_loss = (loss_aim * 100.0) + loss_click

    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Update Learning Rate based on how well we are doing
    scheduler.step(loss_aim)

    if step % 1000 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Step {step} | Loss: {total_loss.item():.5f} | Aim: {loss_aim.item():.5f} | LR: {current_lr:.1e}")

    # Periodic Save
    if step % 10000 == 0 and step > 0:
        torch.save(model.state_dict(), f"osu_model_v9-fixed_step_{step}.pth")

torch.save(model.state_dict(), "osu_model_v9-big.pth")
print("Training Complete.")

Training on: cuda
Found 17999 files in V2 folder.
Training on: cuda
Starting Long-Run Training (Slider Aware)...


Training Final Model:   0%|                                   | 1/100000 [00:00<11:23:49,  2.44it/s]

Step 0 | Loss: 6.77064 | Aim: 0.03962 | LR: 1.0e-03


Training Final Model:   0%|                                  | 46/100000 [00:19<11:36:35,  2.39it/s]


KeyboardInterrupt: 

In [7]:
from torch.utils.data import Dataset, DataLoader

class OsuRamDataset(Dataset):
    def __init__(self, ram_data, seq_len):
        self.data = ram_data
        self.seq_len = seq_len

    def __len__(self):
        # We define "length" arbitrarily since we sample randomly.
        # Making it large allows the DataLoader to run continuously without resetting often.
        return 100000

    def __getitem__(self, idx):
        # --- CRITICAL FIX: Infinite Retry ---
        # Never return bad data (zeros). Keep trying until we find a valid slice.
        while True:
            # Pick random replay
            rand_idx = np.random.randint(0, len(self.data))
            features, labels = self.data[rand_idx]

            if len(features) <= self.seq_len + 5: continue

            # Try 10 random spots in this file
            for _ in range(10):
                start_idx = np.random.randint(0, len(features) - self.seq_len)

                # Copy is essential because we modified the array in RAM loading?
                # Actually, slicing numpy creates a view, but since we convert to Tensor
                # and move to GPU immediately, 'copy' is safer to prevent memory overlaps.
                x_slice = features[start_idx : start_idx + self.seq_len].copy()

                # CPU Checks (Fast)
                # 1. Spinners (Type > 1.5)
                if np.any(x_slice[:, 7] > 1.5): continue
                # 2. Breaks (Time > 5.0)
                if np.max(x_slice[:, 6]) > 5.0: continue

                # Get Labels
                y_slice = labels[start_idx : start_idx + self.seq_len]
                next_pos = y_slice[:, 0:2]
                clicks = y_slice[:, 2:3]

                return x_slice, next_pos, clicks

        # Fallback if 10 tries fail (Should be rare)
        return np.zeros((self.seq_len, 10), dtype=np.float32), \
               np.zeros((self.seq_len, 2), dtype=np.float32), \
               np.zeros((self.seq_len, 1), dtype=np.float32)

In [ ]:
import torch.cuda.amp as amp

# --- OPTIMIZATION FLAGS ---
torch.backends.cuda.matmul.allow_tf32 = True # RTX 4090 Specific Speedup
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# --- CONFIG ---
BATCH_SIZE = 2048
SEQ_LEN = 300
NUM_EPOCHS = 100 # We iterate epochs now because of DataLoader
INITIAL_LR = 1e-4
MAX_LR = 1e-3
FINAL_LR = 1e-6

# Setup Data Loader (Uses multiple CPU cores)

dataset = OsuIndexedDataset(ram_feats, ram_labels, valid_indices, SEQ_LEN)

# Linux Optimization:
# num_workers: Use number of physical cores (not threads). 16 is good.
# prefetch_factor: Keep the queue full.
# persistent_workers: Keeps the threads alive between epochs (Critical for speed).
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)


# Model & Optimizer
device = torch.device("cuda")
model = OsuAimModel(hidden_size=1024).to(device)
optimizer = optim.Adam(model.parameters(), lr=INITIAL_LR)
scaler = amp.GradScaler() # Mixed Precision Scaler

# Loss
pos_weight = torch.tensor([5.0]).to(device)
aim_criterion = nn.L1Loss()
click_criterion = AsymmetricClickLoss(pos_weight=5.0, penalty_for_late=5.0).to(device)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    total_steps=200000,
    pct_start=0.1,  # Warmup for the first 10% (20k steps)
    div_factor=10,  # Start at MAX_LR / 10
    final_div_factor=1000, # End at MAX_LR / 1000
    anneal_strategy='cos'  # Smooth cosine curve
)

print("Starting RTX 4090 Optimized Training...")

step = 0
total_steps = 200000

pbar = tqdm(total=total_steps, desc="Training", ncols=100)

while step < total_steps:
    for batch_x, batch_y_pos, batch_y_click in loader:
        if step >= total_steps: break

        # Move to GPU (Non-blocking)
        x = batch_x.to(device, non_blocking=True)
        y_pos = batch_y_pos.to(device, non_blocking=True)
        y_click = batch_y_click.to(device, non_blocking=True)

        # --- GPU AUGMENTATION (The Speedup Secret) ---
        # We do the math here because the 4090 is 100x faster at it than the CPU
        with torch.no_grad():
            # 1. Normalize Time (Col 6) - Do this first
            # We assume Col 6 is time_delta in seconds (or normalized)

            # --- NEW: TIME NOISE INJECTION ---

            # A. Sequence Offset (Simulate Global Offset / Audio Latency)
            # We shift the WHOLE sequence by a random amount (-15ms to +15ms)
            # Shape: (Batch, 1) -> Broadcast to (Batch, Seq)
            # 0.015 = 15ms
            offset_noise = torch.randn(x.size(0), 1, device=device) * 0.015
            x[:, :, 6] += offset_noise

            # B. Frame Jitter (Simulate Lag Spikes / Polling Rate issues)
            # We jitter EVERY frame independently (-5ms to +5ms)
            # 0.005 = 5ms
            jitter_noise = torch.randn_like(x[:, :, 6]) * 0.005
            x[:, :, 6] += jitter_noise

            # C. Clamp Time
            # Ensure we don't go below 0 (unless you want to allow negative time,
            # but usually LSTM likes 0-1 range for this feature).
            # If your logic allows negative time (late hit), clamp differently.
            # Assuming strictly positive lookahead here:
            x[:, :, 6] = torch.clamp(x[:, :, 6], 0.0, 1.0)

            # 2. Add Noise to Position (Col 0, 1)
            noise = torch.randn_like(x[:, :, 0:2]) * 0.02
            x[:, :, 0:2] += noise
            x[:, :, 0:2] = torch.clamp(x[:, :, 0:2], -0.5, 1.5)

            # 3. Recalculate Deltas (Col 4, 5)
            # Delta = Target(2,3) - Current(0,1)
            x[:, :, 4] = x[:, :, 2] - x[:, :, 0]
            x[:, :, 5] = x[:, :, 3] - x[:, :, 1]

            # 4. Clamp Labels
            y_pos = torch.clamp(y_pos, 0.0, 1.0)

        # --- FORWARD PASS (Mixed Precision) ---
        optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()

        with amp.autocast():
            pred_vel, pred_click, _ = model(x)

            # Physics Step
            curr_pos = x[:, :, 0:2]
            pred_pos = curr_pos + pred_vel

            loss_aim = aim_criterion(pred_pos, y_pos)
            loss_click = click_criterion(pred_click, y_click)

            total_loss = (loss_aim * 100.0) + loss_click

        # 1. Backward Pass
        scaler.scale(total_loss).backward()

        # 2. Update Weights
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # 3. Update Scheduler (EVERY BATCH)
        # This moves the LR along the curve
        scheduler.step()

        # Logging
        if step % 1000 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            # You will see LR go UP for the first 20k steps, then DOWN for the rest.
            tqdm.write(f"Step {step} | Loss: {total_loss.item():.5f} | LR: {current_lr:.2e}")
            pbar.set_postfix({'Loss': f"{total_loss.item():.4f}", 'Aim': f"{loss_aim.item():.4f}", 'LR': f"{current_lr:.1e}"})

            # Periodic Save
            if step % 10000 == 0 and step > 0:
                torch.save(model.state_dict(), f"osu_model_v9_2_step_{step}.pth")

        step += 1
        pbar.update(1)

torch.save(model.state_dict(), "osu_model_v9_final_2.pth")
print("Training Complete.")

/tmp/ipykernel_7484/2478017250.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler() # Mixed Precision Scaler


Starting RTX 4090 Optimized Training...


Training:   0%|                                                          | 0/200000 [00:00<?, ?it/s]/tmp/ipykernel_7484/2478017250.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
/root/miniconda3/envs/gputrain/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Training:   0%|        | 1/200000 [00:21<1184:41:20, 21.32s/it, Loss=6.0921, Aim=0.0328, LR=1.0e-04]

Step 0 | Loss: 6.09213 | LR: 1.00e-04


Training:   0%|         | 64/200000 [00:40<15:37:39,  3.55it/s, Loss=6.0921, Aim=0.0328, LR=1.0e-04]

KeyboardInterrupt: 

In [ ]:
import torch.cuda.amp as amp

# --- OPTIMIZATION FLAGS ---
torch.backends.cuda.matmul.allow_tf32 = True # RTX 4090 Specific Speedup
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
# --- CONFIGURATION ---
# "Physical": What fits in VRAM. 512 is usually safe for 1024-hidden on 4090.
# If this still OOMs, drop it to 256.
PHYSICAL_BATCH_SIZE = 512

# "Effective": The mathematical batch size for stability.
TARGET_BATCH_SIZE = 2048

# Calculate accumulation steps (2048 / 512 = 4 steps)
ACCUMULATION_STEPS = TARGET_BATCH_SIZE // PHYSICAL_BATCH_SIZE

SEQ_LEN = 300
# Total physical iterations to run
TOTAL_PHYSICAL_STEPS = 250000

INITIAL_LR = 1e-4
MAX_LR = 1e-3

print(f"Training Config: Physical {PHYSICAL_BATCH_SIZE} | Accumulate {ACCUMULATION_STEPS}x | Effective {TARGET_BATCH_SIZE}")

# --- SETUP ---
# Use the Fast RAM Dataset (OsuIndexedDataset)
# Note: RAM_DATASET, valid_indices should be loaded from the previous step
dataset = OsuIndexedDataset(ram_feats, ram_labels, valid_indices, SEQ_LEN)

loader = DataLoader(
    dataset,
    batch_size=PHYSICAL_BATCH_SIZE, # Load small batches
    shuffle=True,
    num_workers=16, # High workers is fine on Linux/RAM dataset
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

device = torch.device("cuda")
model = OsuAimModel(input_size=10, hidden_size=1024).to(device)
optimizer = optim.Adam(model.parameters(), lr=INITIAL_LR)
scaler = amp.GradScaler()

aim_criterion = nn.L1Loss(reduction='none')
click_criterion = ExponentialAsymmetricClickLoss(pos_weight=5.0, base_penalty=25.0, expo_factor=5.0).to(device)

# Scheduler steps once per OPTIMIZER update, not per physical batch
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    total_steps=TOTAL_PHYSICAL_STEPS // ACCUMULATION_STEPS,
    pct_start=0.1,
    anneal_strategy='cos'
)

print("Starting Training with Gradient Accumulation...")

# --- TRAINING LOOP ---
optimizer.zero_grad(set_to_none=True)
data_iter = iter(loader)
pbar = tqdm(range(TOTAL_PHYSICAL_STEPS), desc="Training", ncols=120)

for step in pbar:
    # 1. Get Data
    try:
        batch_x, batch_y_pos, batch_y_click = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch_x, batch_y_pos, batch_y_click = next(data_iter)

    x = batch_x.to(device, non_blocking=True)
    y_pos = batch_y_pos.to(device, non_blocking=True)
    y_click = batch_y_click.to(device, non_blocking=True)

    # 2. GPU Augmentation
    # --- GPU AUGMENTATION ---
    with torch.no_grad():
        # 1. Time Manipulations ------------------------------------------------

        # A. Add Jitter & Offset
        x[:, :, 6] += torch.randn(x.size(0), 1, device=device) * 0.015
        x[:, :, 6] += torch.randn_like(x[:, :, 6]) * 0.005

        # B. Apply Time Warp (Speed Up) to 50% of batches
        if np.random.random() < 0.5:
            # Multiplier: 1.1x to 1.5x
            speed_mod = 1.1 + (torch.rand(x.size(0), 1, device=device) * 0.4)
            x[:, :, 6] /= speed_mod

        # C. Final Time Clamp (Do this last to catch everything)
        x[:, :, 6].clamp_(0.0, 1.0)

        # 2. Position Manipulations --------------------------------------------

        # A. Add Noise
        noise = torch.randn_like(x[:, :, 0:2]) * 0.02
        x[:, :, 0:2].add_(noise).clamp_(-0.5, 1.5)

        # B. Recalculate Deltas (Critical: Update distance info based on noise)
        x[:, :, 4] = x[:, :, 2] - x[:, :, 0]
        x[:, :, 5] = x[:, :, 3] - x[:, :, 1]

        # 3. Label Manipulations -----------------------------------------------
        y_pos.clamp_(0.0, 1.0)

    # 3. Forward Pass (Mixed Precision)
    with torch.cuda.amp.autocast():
        pred_vel, pred_click, _ = model(x)
        curr_pos = x[:, :, 0:2]
        pred_pos = curr_pos + pred_vel

        # 1. Physics-Gated Aim Loss
        # Calculate raw error per coordinate
        raw_aim_loss = aim_criterion(pred_pos, y_pos) # (Batch, Seq, 2)

        # Get Click Intensity (0.0 to 1.0)
        click_weight = y_click  # (Batch, Seq, 1)

        # Dynamic Weight: 1.0 (Travel) -> 21.0 (Clicking)
        # We broadcast (Batch, Seq, 1) to (Batch, Seq, 2) automatically
        dynamic_aim_weight = 1.0 + (click_weight * 20.0)

        # Apply Weight and Mean
        loss_aim = (raw_aim_loss * dynamic_aim_weight).mean()

        # 2. Click Loss
        loss_click = click_criterion(pred_click, y_click)

        # 3. Total Weighted Loss
        total_loss = (loss_aim * 50.0) + (loss_click * 1.0)

        # Normalize for Accumulation
        loss_scaled = total_loss / ACCUMULATION_STEPS

    # --- BACKWARD ---
    scaler.scale(loss_scaled).backward()

    # 5. Optimizer Step (Only every 4th step)
    if (step + 1) % ACCUMULATION_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        # Scheduler updates here
        scheduler.step()

    # Logging
    if step % 500 == 0:
        lr = optimizer.param_groups[0]['lr']
        # Show unscaled loss for readability
        pbar.set_postfix({'L': f"{total_loss.item():.4f}", 'Aim': f"{loss_aim.item():.4f}", 'LR': f"{lr:.1e}"})

    if step % 20000 == 0 and step > 0:
        torch.save(model.state_dict(), f"osu_model_4090_step_{step}.pth")

torch.save(model.state_dict(), "osu_model_4090_final.pth")

Training Config: Physical 16 | Accumulate 128x | Effective 2048
Starting Training with Gradient Accumulation...


/tmp/ipykernel_11014/2426847572.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Training:   0%|                                                                              | 0/500000 [00:00<?, ?it/s]/tmp/ipykernel_11014/2426847572.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Training:   1%|▍                              | 6196/500000 [01:06<1:28:35, 92.89it/s, L=4.7199, Aim=0.0527, LR=7.3e-05]